In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import numpy as np
import pandas as pd
from data.market_data import get_multiple_tickers
from pricing.american import american_put_without_dividends, american_put_with_dividends, american_call_without_dividends, american_call_with_dividends
from pricing.european import heston_european_call_option, heston_european_put_option
from calibration.implied_vol import implied_volatility


##### tickers and market constraint

In [2]:
tickers = ['AAPL', 'NVDA']
r = 0.05
q=0
N= 10000
M = 100
spread_limit = 0.05

In [3]:
options_df = get_multiple_tickers(tickers, spread_limit=spread_limit)

pulling...AAPL...
pulling...NVDA...


In [4]:
options_df.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,type,maturity,T,spot,ticker,ExerciseStyle
0,AAPL260302C00190000,2026-03-02 19:49:28+00:00,190.0,75.83,73.40,75.95,2.580002,3.522187,3.0,29.0,4.074224,True,REGULAR,USD,call,2026-03-02,-0.00274,264.720001,AAPL,american
1,AAPL260302C00205000,2026-03-02 19:51:26+00:00,205.0,61.06,58.25,60.95,-8.439999,-12.143883,7.0,3.0,3.299806,True,REGULAR,USD,call,2026-03-02,-0.00274,264.720001,AAPL,american
2,AAPL260302C00250000,2026-03-02 20:27:16+00:00,250.0,14.35,13.90,14.05,-1.030000,-6.697007,23.0,22.0,0.000010,True,REGULAR,USD,call,2026-03-02,-0.00274,264.720001,AAPL,american
3,AAPL260302C00252500,2026-03-02 19:28:43+00:00,252.5,13.41,11.55,11.75,0.559999,4.357972,41.0,25.0,0.000010,True,REGULAR,USD,call,2026-03-02,-0.00274,264.720001,AAPL,american
4,AAPL260302C00255000,2026-03-02 20:10:20+00:00,255.0,9.80,8.85,9.00,-1.700000,-14.782607,74.0,64.0,0.000010,True,REGULAR,USD,call,2026-03-02,-0.00274,264.720001,AAPL,american


#### Filtering for one option only to check for model flow

In [5]:
options_df_nvda = options_df[(options_df['ticker']=='NVDA') & (options_df['maturity']=='2026-04-17') &(options_df['strike'] == 200) & (options_df['type']=='call')]
options_df_nvda

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,type,maturity,T,spot,ticker,ExerciseStyle
2022,NVDA260417C00200000,2026-03-02 20:42:30+00:00,200.0,4.67,4.6,4.7,0.82,21.298706,13522.0,48061.0,0.417669,False,REGULAR,USD,call,2026-04-17,0.123288,182.373001,NVDA,american


##### Testing on fixed heston parameters for now

In [6]:
heston_params = (0.04, 2.0, 0.04, 0.5, -0.7)

In [7]:
def price_row(row, r, q, heston_params ):
    S0 = row['spot']
    K=row['strike']
    T= row['T']
    option_type = row['type']

    v0, kappa, theta, sigma, rho = heston_params

    if option_type == 'call':
        if q==0:
            print('american_call_without_dividends ....exexcuted')
            return american_call_without_dividends(S0, K, T, r, v0, kappa, theta, sigma, rho)
        else:
            print('american_call_with_dividends....exexcuted')
            return american_call_with_dividends(S0, K, T, r, v0, kappa, theta, sigma, rho, M, N, q)
    elif option_type == 'put':
        if q==0:
            print('american_put_without_dividends....exexcuted')
            return american_put_without_dividends(S0, K, T, r, v0, kappa, theta, sigma, rho, M, N)
        else:
            print('american_put_with_dividends....exexcuted')
            return american_put_with_dividends(S0, K, T, r, v0, kappa, theta, sigma, rho, M, N, q)

In [8]:
options_df_nvda['heston_price'] = options_df_nvda.apply(lambda row: price_row(row, r, q, heston_params), axis=1)

american_call_without_dividends ....exexcuted


In [9]:
options_df_nvda.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,inTheMoney,contractSize,currency,type,maturity,T,spot,ticker,ExerciseStyle,heston_price
2022,NVDA260417C00200000,2026-03-02 20:42:30+00:00,200.0,4.67,4.6,4.7,0.82,21.298706,13522.0,48061.0,...,False,REGULAR,USD,call,2026-04-17,0.123288,182.373001,NVDA,american,0.295318


##### Implied volatility using BS

In [10]:
options_df_nvda["market_iv"] = options_df_nvda.apply(
    lambda row: implied_volatility(
        row["heston_price"],
        row["spot"],
        row["strike"],
        row["T"],
        r,
        row["type"],
        q
    ),
    axis=1
)
options_df_nvda.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,contractSize,currency,type,maturity,T,spot,ticker,ExerciseStyle,heston_price,market_iv
2022,NVDA260417C00200000,2026-03-02 20:42:30+00:00,200.0,4.67,4.6,4.7,0.82,21.298706,13522.0,48061.0,...,REGULAR,USD,call,2026-04-17,0.123288,182.373001,NVDA,american,0.295318,0.160541
